# Resume Role Prediction with Fine-Tuned GPT

This project demonstrates how to fine-tune a GPT 4.1-nano model to predict job roles from synthetic resume summaries.

## Project Overview
I leverage a curated dataset of synthetic resume summaries and corresponding job roles to train a GPT-based model. The goal is to have the model predict the job role given a resume summary, which can be tested interactively via a Gradio interface.



In [61]:
# imports

import os
import json
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import random
import gradio as gr
import seaborn as sns
from sklearn.metrics import confusion_matrix
from openai import OpenAI
load_dotenv(override=True)

True

In [ ]:


hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [10]:
openai = OpenAI()

In [5]:
dataset = load_dataset("VictorHabila/synthetic_resume_summaries", split="train", trust_remote_code=True)

In [ ]:
print(f"Number of synthetic resume_summaries: {len(dataset):,}")

In [ ]:
dataset[0]

In [12]:
def messages_for_resume(row):
    message = f"Given this resume summary, predict the job role. Respond with the role, no explanation.\n\n{row['job_summary']}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"{row['actual_role']}"}
    ]



In [13]:
def make_jsonl(df_subset):
    result = ""
    for _, row in df_subset.iterrows():
        messages = messages_for_resume(row)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str + '}\n'
    return result.strip()



In [14]:
def write_jsonl(df_subset, filename):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(make_jsonl(df_subset))

In [ ]:
# Convert to a Pandas DataFrame
df = pd.DataFrame(dataset)
print(f"Loaded {len(df):,} entries")
df.head()

In [22]:
os.makedirs("jsonl", exist_ok=True)

In [23]:
fine_tune_train = df[:100]
fine_tune_validation = df[100:150]

write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [24]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")


In [25]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
print(train_file.id, validation_file.id)

In [29]:
fine_tune_job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="resume_role_predictor"
)

In [ ]:
openai.fine_tuning.jobs.list(limit=1)

In [32]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [ ]:
job_id

In [ ]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

In [35]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
fine_tuned_model_name

In [45]:
def test_messages_for_resume(row):
    message = f"Given this resume summary, predict the job role. Respond with the role, no explanation.\n\n {row['job_summary']}"
    return [{"role": "user", "content": message}]



In [ ]:
test_messages_for_resume(df.iloc[1])

In [47]:
def predict_role(row):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for_resume(row),
        max_tokens=20
    )
    return response.choices[0].message.content.strip()

In [ ]:
row = df.iloc[3]  # Pick one test row
print(row)
print("Actual:", row['actual_role'])
print("Predicted:", predict_role(row))

In [71]:
test_df = df[150:200]  # for example
correct = 0
for _, row in test_df.iterrows():
    if predict_role(row) == row['actual_role']:
        correct += 1

accuracy = correct / len(test_df) * 100
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 8.00%


## RESULT VISUALIZATION

In [ ]:

%matplotlib inline

predicted_roles = [predict_role(row) for row in test_df.to_dict(orient="records")]
actual_roles = test_df['actual_role'].tolist()


all_roles = sorted(list(set(actual_roles + predicted_roles)))

cm = confusion_matrix(actual_roles, predicted_roles, labels=all_roles)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks(np.arange(len(all_roles)))
ax.set_yticks(np.arange(len(all_roles)))
ax.set_xticklabels(all_roles, rotation=45, ha="right")
ax.set_yticklabels(all_roles)

ax.set_xlabel("Predicted Role")
ax.set_ylabel("Actual Role")
ax.set_title("Confusion Matrix of Resume Role Predictions")

for i in range(len(all_roles)):
    for j in range(len(all_roles)):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")

plt.tight_layout()
plt.show()

accuracy_per_class = {}
for role in all_roles:
    idx = [j for j, val in enumerate(actual_roles) if val == role]
    if len(idx) == 0:
        continue  
    acc = sum([actual_roles[j] == predicted_roles[j] for j in idx]) / len(idx) * 100
    accuracy_per_class[role] = acc

plt.figure(figsize=(12, 5))
plt.bar(accuracy_per_class.keys(), accuracy_per_class.values(), color='skyblue')
plt.xticks(rotation=45, ha='right')
plt.ylabel("Accuracy (%)")
plt.title("Prediction Accuracy per Job Role")
plt.show()

In [ ]:
import gradio as gr

def gradio_predict(summary):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=[{"role": "user", "content": f"Given this resume summary, predict the job role. Respond with the role, no explanation:\n\n{summary}"}]
    )
    return response.choices[0].message.content.strip()

iface = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(lines=8, placeholder="Paste a resume summary here..."),
    outputs="text",
    title="Resume Role Predictor",
    description="Paste a resume summary and get the predicted job role."
)

iface.launch()